In [13]:
import os

!pip install pyspark

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver", exist_ok=True)

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnull, trim, to_date

spark = SparkSession.builder.appName("DataLakePreprocessing").getOrCreate()

bronze_df = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)

bronze_df.printSchema()
bronze_df.show(5)

initial_count = bronze_df.count()
print(f"Total Bronze Records: {initial_count}")

bronze_df.select([count(when(isnull(c), c)).alias(c) for c in bronze_df.columns]).show()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

In [15]:
from pyspark.sql.functions import col, trim, expr

spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

silver_df = bronze_df.dropDuplicates()

string_cols = [f.name for f in silver_df.schema.fields if str(f.dataType) == "StringType"]
for c in string_cols:
    silver_df = silver_df.withColumn(c, trim(silver_df[c]))

silver_df = silver_df.withColumn(
    "order_date",
    expr("coalesce(to_date(order_date, 'yyyy-MM-dd'), to_date(order_date, 'dd/MM/yyyy'))")
)

silver_df.coalesce(1).write.csv("data_lake/silver/preprocessed_sales", header=True, mode="overwrite")

In [16]:
final_count = silver_df.count()
print(f"Bronze Count: {initial_count} | Silver Count: {final_count}")

silver_check = spark.read.csv("data_lake/silver/preprocessed_sales", header=True, inferSchema=True)
silver_check.show(5)

Bronze Count: 1000 | Silver Count: 990
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|       670|              30|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|       670|               0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      C0228|     Diya Tripathi|Competitive Exam ...|    Books|       6|       610